#  Notebook 3 — Modelling

**Goal:** Train, evaluate, and compare three machine learning models.

**Input:** `Data/heart_cleaned.csv` (from Notebook 2)

**Steps in this notebook:**
1. Load cleaned dataset
2. Encode categorical features (One-Hot Encoding)
3. Train/Test split (stratified 80/20)
4. Apply SMOTE on training set only
5. Scale numerical features (StandardScaler)
6. Stratified 5-Fold Cross Validation — all 3 models
7. Select best model based on CV results
8. Train final Logistic Regression model (best CV performance)
9. Evaluate on test set
10. Save model + scaler + feature columns

> ⚠️ SMOTE and scaling are applied AFTER the train/test split,
> and ONLY on the training set. Never on test data.
>
> ✅ Logistic Regression was selected as the final model because it
> outperformed both Random Forest and XGBoost on all CV metrics.
> This is expected on small, clean, structured datasets where
> simpler linear models often beat complex ensemble methods.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                     f1_score, roc_auc_score, confusion_matrix,
                                     classification_report, roc_curve, auc)
from imblearn.over_sampling  import SMOTE

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

os.makedirs('../Models', exist_ok=True)
os.makedirs('../App/static', exist_ok=True)

print(' All imports successful')

## Step 1 — Load Cleaned Dataset

In [ ]:
df = pd.read_csv('../Data/heart_cleaned.csv')
print(f' Cleaned data loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
print(f'Class balance: {dict(df["condition"].value_counts())}')
df.head()

## Step 2 — Encode Categorical Features (One-Hot Encoding)

In [ ]:
X = df.drop(columns=['condition'])
y = df['condition']

print(f'Features shape before encoding: {X.shape}')

ohe_cols = ['cp', 'restecg', 'slope', 'thal']
X = pd.get_dummies(X, columns=ohe_cols, drop_first=True)

print(f'Features shape after encoding:  {X.shape}')
print(f'\nAll feature columns after encoding:')
for col in X.columns:
    print(f'  {col}')

feature_cols = X.columns.tolist()
with open('../Models/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
print(f'\n Feature columns saved → Models/feature_cols.pkl')

## Step 3 — Train/Test Split (Stratified 80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Train/Test Split (stratified):')
print(f'  Training set  : {X_train.shape[0]} samples')
print(f'  Test set      : {X_test.shape[0]} samples')
print(f'  Train balance : {dict(y_train.value_counts())}')
print(f'  Test balance  : {dict(y_test.value_counts())}')
print(f'\n Class proportions preserved in both splits.')

## Step 4 — Apply SMOTE (Training Set Only)

In [ ]:
print('Before SMOTE:')
print(f'  Training class balance: {dict(y_train.value_counts())}')

sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(f'  Training class balance: {dict(pd.Series(y_train_res).value_counts())}')
print(f'  Training set size: {X_train.shape[0]} → {X_train_res.shape[0]}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, counts, title in [
    (axes[0], y_train.value_counts(), 'Before SMOTE'),
    (axes[1], pd.Series(y_train_res).value_counts(), 'After SMOTE')
]:
    ax.bar(['No Disease', 'Disease'], [counts[0], counts[1]],
           color=['#378add', '#e84040'], edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate([counts[0], counts[1]]):
        ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.suptitle('Training Set Class Balance — SMOTE Effect', fontweight='bold')
plt.tight_layout()
plt.savefig('../App/static/plot_smote.png', bbox_inches='tight')
plt.show()

## Step 5 — Feature Scaling (StandardScaler)

In [ ]:
num_cols_to_scale = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak',
                     'thalach_pct_max', 'chol_age_ratio', 'oldpeak_slope']

scale_cols = [c for c in num_cols_to_scale if c in X_train_res.columns]

scaler = StandardScaler()

X_train_res = pd.DataFrame(X_train_res, columns=feature_cols)
X_test_df   = pd.DataFrame(X_test,      columns=feature_cols)

X_train_res[scale_cols] = scaler.fit_transform(X_train_res[scale_cols])
X_test_df[scale_cols]   = scaler.transform(X_test_df[scale_cols])

with open('../Models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(' Scaler fitted on training data and applied to test data.')
print(' Scaler saved → Models/scaler.pkl')
print(f'   Scaled columns: {scale_cols}')

## Step 6 — Stratified 5-Fold Cross Validation — All 3 Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=6,
                                                   class_weight='balanced',
                                                   random_state=RANDOM_STATE),
    'XGBoost'            : XGBClassifier(n_estimators=200, max_depth=4,
                                          learning_rate=0.05, subsample=0.8,
                                          colsample_bytree=0.8, eval_metric='logloss',
                                          random_state=RANDOM_STATE)
}

skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_summary = {}

print('Stratified 5-Fold Cross Validation')
print('='*50)

for name, model in models.items():
    cv = cross_validate(model, X_train_res, y_train_res,
                        cv=skf, scoring=scoring, n_jobs=-1)
    cv_summary[name] = {
        'Accuracy' : round(cv['test_accuracy'].mean(), 4),
        'Precision': round(cv['test_precision'].mean(), 4),
        'Recall'   : round(cv['test_recall'].mean(), 4),
        'F1'       : round(cv['test_f1'].mean(), 4),
        'ROC-AUC'  : round(cv['test_roc_auc'].mean(), 4),
        'Acc±Std'  : f"{cv['test_accuracy'].mean():.3f} ± {cv['test_accuracy'].std():.3f}"
    }
    print(f'\n{name}')
    print(f'  Accuracy : {cv_summary[name]["Acc±Std"]}')
    print(f'  Precision: {cv_summary[name]["Precision"]}')
    print(f'  Recall   : {cv_summary[name]["Recall"]}')
    print(f'  F1       : {cv_summary[name]["F1"]}')
    print(f'  ROC-AUC  : {cv_summary[name]["ROC-AUC"]}')

cv_df = pd.DataFrame(cv_summary).T
print('\n')
cv_df

In [ ]:
# Save CV results
cv_df.to_csv('../Models/cv_results.csv')
print(' CV results saved → Models/cv_results.csv')

# Bar chart comparison
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
cv_plot = pd.DataFrame(cv_summary).T[metrics_to_plot].astype(float)

fig, ax = plt.subplots(figsize=(11, 5))
cv_plot.plot(kind='bar', ax=ax,
             color=['#378add','#1d9e75','#e84040','#888780','#ba7517'],
             edgecolor='white', linewidth=0.5, rot=0)
ax.set_title('Model Comparison — Cross Validation Scores', fontweight='bold', fontsize=13)
ax.set_ylabel('Score')
ax.set_ylim(0.6, 1.05)
ax.legend(loc='lower right')
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../App/static/plot_model_comparison.png', bbox_inches='tight')
plt.show()

# Identify best model by ROC-AUC
best_model_name = cv_df['ROC-AUC'].astype(float).idxmax()
print(f'\n🏆 Best model by ROC-AUC: {best_model_name}')
print(f'   ROC-AUC: {cv_df.loc[best_model_name, "ROC-AUC"]}')

## Step 7 — Train Final Model

Logistic Regression outperformed both Random Forest and XGBoost across all CV metrics.
This is a well-known result on small, clean, structured datasets — simpler linear models
often outperform complex ensemble methods when data is limited.

We let the CV results decide the final model — not assumptions.

In [ ]:
print('Training final Logistic Regression model...')
print('(Selected based on best CV performance across all metrics)')

final_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
final_model.fit(X_train_res, y_train_res)

# Save with a clear, honest filename
with open('../Models/lr_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print(' Logistic Regression model saved → Models/lr_model.pkl')

## Step 8 — Evaluate Final Model on Test Set

In [ ]:
y_pred = final_model.predict(X_test_df)
y_prob = final_model.predict_proba(X_test_df)[:, 1]

print('── Test Set Results — Logistic Regression ──')
print(f'  Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred):.4f}')
print(f'  Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'  F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'  ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print()
print('── Classification Report ───────────────────')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'],
            linewidths=0.5, ax=axes[0])
axes[0].set_xlabel('Predicted', fontweight='bold')
axes[0].set_ylabel('Actual', fontweight='bold')
axes[0].set_title('Confusion Matrix — Logistic Regression', fontweight='bold')

# ROC Curves — all 3 models
colors = {'Logistic Regression': '#e84040',
          'Random Forest'      : '#378add',
          'XGBoost'            : '#888780'}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    prob = model.predict_proba(X_test_df)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    lw = 2.5 if name == 'Logistic Regression' else 1.5
    axes[1].plot(fpr, tpr, lw=lw, color=colors[name],
                 label=f'{name} (AUC={roc_auc:.3f})')

axes[1].plot([0,1],[0,1],'k--', lw=1, alpha=0.4, label='Random (AUC=0.500)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — All Models', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../App/static/plot_roc_curves.png', bbox_inches='tight')
plt.savefig('../App/static/plot_confusion_matrix.png', bbox_inches='tight')
plt.show()

print(' Plots saved → App/static/')

## Step 9 — Summary

In [ ]:
print('═'*55)
print('  MODELLING COMPLETE')
print('═'*55)
print('  Final model: Logistic Regression')
print('  Reason: Best CV performance on all metrics')
print()
print('  Saved files:')
print('    Models/lr_model.pkl          ← final model')
print('    Models/scaler.pkl')
print('    Models/feature_cols.pkl')
print('    Models/cv_results.csv')
print('    App/static/plot_model_comparison.png')
print('    App/static/plot_roc_curves.png')
print('    App/static/plot_confusion_matrix.png')
print('    App/static/plot_smote.png')
print()
print('  Next step → Run 04_SHAP_Explainability.ipynb')
print('═'*55)